# Proyecto 2 — Análisis Exploratorio de Datos

## Reto #11: Predicción de compradores recurrentes: cuestionar la línea base

**CC3084 – Data Science | Universidad del Valle de Guatemala | Semestre II – 2026**

**David Dominguez - 23712**
**Gabriel Bran - 23590**
**Luis Padilla - 23663**


---

### Alcance

Este notebook contiene un **análisis exploratorio de datos (EDA)** completo sobre el conjunto de datos de la competencia *Repeat Buyers Prediction* de la plataforma Tianchi (Alibaba). **No se entrenan modelos predictivos** en este proyecto.

### Pregunta de análisis

> ¿Qué patrones de comportamiento de compra y características demográficas se asocian con que un usuario de la plataforma Tmall vuelva a comprar a un mismo vendedor?

### Enlace al reto original

[Tianchi — Repeat Buyers Prediction](https://tianchi.aliyun.com/competition/entrance/231576/information)

## 2. Configuración

In [3]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
import gc

# Suprimir advertencias innecesarias
warnings.filterwarnings('ignore')

# Semilla para reproducibilidad
np.random.seed(42)

# --- Rutas del proyecto, resueltas desde la raíz real del repositorio ---
def _find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'data' / 'data_format1').exists():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / 'data' / 'data_format1'
TRAIN_PATH = DATA_DIR / 'train_format1.csv'
TEST_PATH = DATA_DIR / 'test_format1.csv'
USER_INFO_PATH = DATA_DIR / 'user_info_format1.csv'
USER_LOG_PATH = DATA_DIR / 'user_log_format1.csv'
SAMPLE_SUB_PATH = PROJECT_ROOT / 'data' / 'sample_submission.csv'

FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures'
TAB_DIR = PROJECT_ROOT / 'outputs' / 'tables'

# --- Crear directorios de salida ---
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

# --- Configuración visual ---
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style('whitegrid')
PALETTE = sns.color_palette('Set2')
COLOR_0 = PALETTE[1]  # No recurrente
COLOR_1 = PALETTE[0]  # Recurrente
labels_map = {0: 'No recurrente', 1: 'Recurrente'}


def normalize_age_range(series: pd.Series) -> pd.Series:
    """Conserva 0 como desconocido y colapsa 7/8 en 50+."""
    return series.fillna(0).replace({8: 7}).astype(int)


age_labels = {
    0: 'Desconocido',
    1: '<18',
    2: '18-24',
    3: '25-29',
    4: '30-34',
    5: '35-39',
    6: '40-49',
    7: '50+',
}
gender_labels = {0: 'Femenino', 1: 'Masculino', 2: 'Desconocido'}

print("Configuración completada.")

Configuración completada.


## 3. Verificación de archivos

In [4]:
archivos = {
    'Entrenamiento': TRAIN_PATH,
    'Prueba': TEST_PATH,
    'Información de usuarios': USER_INFO_PATH,
    'Registro de actividad': USER_LOG_PATH,
    'Envío de muestra': SAMPLE_SUB_PATH,
}

print(f"{'Archivo':<30} {'Existe':<8} {'Tamaño (MB)':>12}")
print("-" * 55)
for nombre, ruta in archivos.items():
    existe = os.path.isfile(ruta)
    if existe:
        tam = os.path.getsize(ruta) / (1024 ** 2)
        print(f"{nombre:<30} {'Sí':<8} {tam:>10.1f} MB")
    else:
        print(f"{nombre:<30} {'NO':<8} {'---':>12}")

print()
print(" NOTA: El archivo de registro de actividad (user_log_format1.csv)")
print("   ocupa aproximadamente 1.9 GB y contiene ~55 millones de filas.")
print("   Se procesará mediante lectura por fragmentos (chunks).")

Archivo                        Existe    Tamaño (MB)
-------------------------------------------------------
Entrenamiento                  Sí              3.4 MB
Prueba                         Sí              3.1 MB
Información de usuarios        Sí              4.3 MB
Registro de actividad          Sí           1821.7 MB
Envío de muestra               Sí              3.9 MB

 NOTA: El archivo de registro de actividad (user_log_format1.csv)
   ocupa aproximadamente 1.9 GB y contiene ~55 millones de filas.
   Se procesará mediante lectura por fragmentos (chunks).


## 4. Carga de datos

Los archivos pequeños (`train`, `test`, `user_info`) se cargan completos en memoria.
El archivo de registros de actividad (`user_log`) contiene ~55 millones de filas (~1.9 GB),
por lo que se procesa mediante lectura por fragmentos (`chunksize`) para evitar problemas de memoria.

### 4.1 Carga de archivos pequeños

In [5]:
# Cargar train, test, user_info
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
user_info_raw = pd.read_csv(USER_INFO_PATH)

print(f"train_format1:     {train_raw.shape[0]:>10,} filas × {train_raw.shape[1]} columnas")
print(f"test_format1:      {test_raw.shape[0]:>10,} filas × {test_raw.shape[1]} columnas")
print(f"user_info_format1: {user_info_raw.shape[0]:>10,} filas × {user_info_raw.shape[1]} columnas")

train_format1:        260,864 filas × 3 columnas
test_format1:         261,477 filas × 3 columnas
user_info_format1:    424,170 filas × 3 columnas


### 4.2 Procesamiento del registro de actividad (user_log)

Dado el tamano del archivo (~55 millones de filas), se procesa en dos pasos para evitar cargarlo completo en memoria.

**Estrategia exacta**: cada fila se asigna a una particion en disco segun `hash(user_id, seller_id) mod N`. Asi, todos los registros de un mismo par quedan en la misma particion. Luego cada particion se agrupa completa por `(user_id, seller_id)` para calcular:
- Conteos de cada tipo de accion (clics, carrito, compras, favoritos)
- Total de acciones
- Conteo de valores unicos de productos, categorias, marcas y dias

Este enfoque es exacto, porque ningun par usuario-vendedor se divide entre particiones, y sigue siendo eficiente porque solo se carga una particion a la vez.

In [6]:
import shutil

CHUNKSIZE = 2_000_000
N_PARTITIONS = 128
PAIR_COLS = ['user_id', 'seller_id']
LOG_COLUMNS = ['user_id', 'seller_id', 'action_type', 'item_id', 'cat_id', 'brand_id', 'time_stamp']
LOG_DTYPE = {
    'user_id': 'int32',
    'seller_id': 'int32',
    'action_type': 'int8',
    'item_id': 'int32',
    'cat_id': 'int32',
    'brand_id': 'float64',
    'time_stamp': 'int16',
}

LOG_TMP_DIR = PROJECT_ROOT / 'outputs' / 'tmp' / 'user_log_partitions'
if LOG_TMP_DIR.exists():
    shutil.rmtree(LOG_TMP_DIR)
LOG_TMP_DIR.mkdir(parents=True, exist_ok=True)

part_files = [LOG_TMP_DIR / f'part_{i:03d}.csv' for i in range(N_PARTITIONS)]
written_headers = [False] * N_PARTITIONS

print('Dispersando user_log_format1.csv en particiones exactas...')
print(f'Fragmento de lectura: {CHUNKSIZE:,} filas')
print(f'Particiones en disco: {N_PARTITIONS}\n')

total_rows = 0
chunk_num = 0

for chunk in pd.read_csv(USER_LOG_PATH, usecols=LOG_COLUMNS, dtype=LOG_DTYPE, chunksize=CHUNKSIZE):
    chunk_num += 1
    total_rows += len(chunk)

    partitions = (
        pd.util.hash_pandas_object(chunk[PAIR_COLS], index=False)
        .to_numpy(dtype='uint64', copy=False) % N_PARTITIONS
    ).astype(np.int16)
    chunk['_partition'] = partitions

    for part_id, part_chunk in chunk.groupby('_partition', sort=False):
        part_id = int(part_id)
        output_path = part_files[part_id]
        part_chunk = part_chunk.drop(columns='_partition')
        part_chunk.to_csv(
            output_path,
            index=False,
            mode='a',
            header=not written_headers[part_id],
        )
        written_headers[part_id] = True

    print(
        f'  Fragmento {chunk_num:>3}: {total_rows:>12,} filas acumuladas '
        f'({total_rows / 54_925_330 * 100:>5.1f}%)'
    )

print('\nAgrupando particiones exactas por par usuario-vendedor...')

log_parts = []
rows_from_partitions = 0

for part_id, part_path in enumerate(part_files, start=1):
    if not part_path.exists() or part_path.stat().st_size == 0:
        continue

    part = pd.read_csv(part_path, dtype=LOG_DTYPE)
    rows_from_partitions += len(part)

    part['is_click'] = (part['action_type'] == 0).astype(np.int32)
    part['is_cart'] = (part['action_type'] == 1).astype(np.int32)
    part['is_purchase'] = (part['action_type'] == 2).astype(np.int32)
    part['is_favorite'] = (part['action_type'] == 3).astype(np.int32)

    agg = part.groupby(PAIR_COLS, as_index=False).agg(
        clicks=('is_click', 'sum'),
        cart=('is_cart', 'sum'),
        purchase=('is_purchase', 'sum'),
        favorite=('is_favorite', 'sum'),
        total_actions=('action_type', 'size'),
        n_items=('item_id', 'nunique'),
        n_cats=('cat_id', 'nunique'),
        n_brands=('brand_id', 'nunique'),
        n_days=('time_stamp', 'nunique'),
    )
    log_parts.append(agg)

    print(
        f'  Particion {part_id:>3}: {len(part):>12,} filas | '
        f'pares unicos: {len(agg):,}'
    )

    del part, agg
    gc.collect()

log_agg = pd.concat(log_parts, ignore_index=True)
log_agg = log_agg.rename(columns={'seller_id': 'merchant_id'})

# Validacion estructural de exactitud
assert rows_from_partitions == total_rows
assert not log_agg[['user_id', 'merchant_id']].duplicated().any()
assert len(log_agg) == log_agg[['user_id', 'merchant_id']].shape[0]

# Liberar archivos temporales
shutil.rmtree(LOG_TMP_DIR, ignore_errors=True)

del log_parts

gc.collect()

print(f"\n{'='*60}")
print(f'Total de filas procesadas: {total_rows:,}')
print(f'Pares unicos en log_agg:   {len(log_agg):,}')
print(f'Memoria de log_agg:        {log_agg.memory_usage(deep=True).sum() / 1024**2:.1f} MB')


Dispersando user_log_format1.csv en particiones exactas...
Fragmento de lectura: 2,000,000 filas
Particiones en disco: 128

  Fragmento   1:    2,000,000 filas acumuladas (  3.6%)
  Fragmento   2:    4,000,000 filas acumuladas (  7.3%)


KeyboardInterrupt: 